In [3]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import numpy as np
from order_separation import OptimizeIntensities
from scipy.special import factorial, ive, binom

In [5]:
def get_nQ_bessel_model(n,*,x0=1,a=1):
    def nQ_bessel_model(x):
        xp = x/x0
        if n == 0:
            f = (ive(0,xp/2) - 1)
        else:
            f = ive(n,xp/2)
        return a*(-1)**n*f
    return nQ_bessel_model

def get_nQ_bessel_series(n,*,x0=1,a=1):
    def nQ_bessel_series(m,x):
        xp = x/x0
        return a*(-1)**(m)*(xp/4)**m/factorial(m)*binom(2*m,m-n)
    return nQ_bessel_series

In [18]:
# quantities taken from TA data
S_max = 1 # maximum TA signal strength
I_sat = 1 # intensity at which signal reaches (1 - 1/e) * S_max
noise_measured = 1E-3 # noise is the standard deviation of the noise of the TA measurement

# input parameters
n = 2 #1 for 1Q, 2 for 2Q, etc.
N = 3 # number of intensities to use
M = 3 # number of orders you care about

intensity_selection = 'any' 
# intensity_selection = 'Intensity Cycling'
# intensity_selection = 'Chebyshev' 
# intensity_selection = 'Linear Sampling'

# calculation
noise = noise_measured / S_max
f_series = get_nQ_bessel_series(n)
f = get_nQ_bessel_model(n)
opt2 = OptimizeIntensities(f,f_series,noise)
res = opt2.minimize_errors_single_N(M,N,intensity_selection=intensity_selection)
print('Optimal intensity values are',np.round(res[0] * I_sat,2))
opt2.print_relative_errors(res[0])

# errors are printed for each order starting from 3rd order
# if the signal should be 0, the "relative" errors are actually absolute errors

[0.28329182 1.13972541 1.77246678]
Optimal intensity values are [0.28 1.14 1.77]
Relative random error for each order [0.0006  0.28038 0.20877] 
 Relative systematic error for each order [1.5000e-04 2.2744e-01 5.9683e-01]
